# Get dataset spatial coverage and temporal resolution

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import xyzservices as xyz
import cartopy
from pyproj import Transformer
import cartopy.crs as ccrs
from cartopy import feature as cfeature
from global_snowmelt_runoff_onset.config import Config, Tile
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import geopandas as gpd
import numpy as np
import dask
import zarr
import tqdm
import pandas as pd
import pickle
import os
from pathlib import Path

In [ ]:
config = Config('config/global_config_v10.txt')

VERSION = config.version
RESULTS_DIR = Path('results') / VERSION
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Load in MODIS dataset and geometries

In [ ]:
modis_seasonal_snow_mask_ds = xr.open_zarr(config.snow_phenology_store, decode_coords="all")
modis_seasonal_snow_mask_ds

In [ ]:
world_gdf = gpd.read_file(
    "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
)
greenland_gdf = world_gdf[world_gdf['ADMIN'] == 'Greenland']
greenland_proj_gdf = greenland_gdf.to_crs(modis_seasonal_snow_mask_ds.rio.crs)
greenland_proj_gdf['geometry'] = greenland_proj_gdf.simplify(100)

In [ ]:
# read from shapefile in geometries folder, in canada_poly.zip
canadian_archipelago_gdf = gpd.read_file("data/canada_poly.zip")
canadian_archipelago_proj_gdf = canadian_archipelago_gdf.to_crs(modis_seasonal_snow_mask_ds.rio.crs)
canadian_archipelago_proj_gdf['geometry'] = canadian_archipelago_proj_gdf.simplify(100)

In [ ]:
russia_gdf = world_gdf[world_gdf['ADMIN'] == 'Russia']
russia_proj_gdf = russia_gdf.to_crs(modis_seasonal_snow_mask_ds.rio.crs)
# filter out all but the largest polygon by area, you can access like geom.area in russia_proj_gdf.geometry.iloc[0].geoms to get the individual polygons, and then calculate area for each and filter out all but the largest one. The largest one is the mainland, the second largest one is the russian arctic archipelago, and the rest are small islands that we can ignore for this analysis.
russia_proj_gdf = russia_proj_gdf.explode(index_parts=False)
russia_proj_gdf['area'] = russia_proj_gdf.geometry.area
russia_proj_gdf = russia_proj_gdf.sort_values('area', ascending=False).reset_index(drop=True)
russia_proj_gdf['number'] = russia_proj_gdf.index
# remove 0 and 1, which are the mainland and the russian arctic archipelago, respectively
russia_proj_gdf = russia_proj_gdf[russia_proj_gdf['number'] > 1]
# remove all below latitude of 70 N, first convert 70N to y location using transform, not cartopy but pyproj
transformer = Transformer.from_crs("EPSG:4326", modis_seasonal_snow_mask_ds.rio.crs, always_xy=True)
y_coord = transformer.transform(0, 70)
russian_archipelago_proj_gdf = russia_proj_gdf[russia_proj_gdf.geometry.centroid.y > y_coord[1]]
russian_archipelago_proj_gdf['geometry'] = russian_archipelago_proj_gdf.simplify(100)

In [ ]:
f,ax=plt.subplots()
greenland_proj_gdf.plot(ax=ax, color='blue')
canadian_archipelago_proj_gdf.plot(ax=ax, color='green')
russian_archipelago_proj_gdf.plot(ax=ax, color='red')

## Get MODIS observable seasonal snow extent

In [ ]:
results_list = []

# Get the MODIS data bounds
modis_ds = modis_seasonal_snow_mask_ds['max_consec_snow_days'].sel(y=slice(None, -0.65E7))

# Get y and x coordinate arrays
y_coords = modis_ds.y.values
x_coords = modis_ds.x.values

# MODIS pixel area in km^2
modis_pixel_area_km2 = (modis_ds.rio.resolution()[0] / 1000) ** 2

# Initialize area tracking
water_year_areas = {wy: 0.0 for wy in modis_seasonal_snow_mask_ds.water_year.values}
median_composite_area = 0.0

# Process in y-coordinate bands to avoid memory issues
y_band_size = 500  # Process 2000 rows at a time
n_y = len(y_coords)

print("\n=== Processing MODIS data (all water years + 10-year composite) ===")

for i in tqdm.tqdm(range(0, n_y, y_band_size)):
    end_idx = min(i + y_band_size, n_y)
    
    # Get this y-coordinate band for ALL water years at once
    y_band_all_years = modis_ds.isel(y=slice(i, end_idx))
    
    # Clip out regions (apply to all years at once)
    y_band_clipped = y_band_all_years.rio.clip(
        greenland_proj_gdf.geometry, 
        drop=False, 
        invert=True
    )
    y_band_clipped = y_band_clipped.rio.clip(
        canadian_archipelago_proj_gdf.geometry,
        drop=False,
        invert=True
    )
    y_band_clipped = y_band_clipped.rio.clip(
        russian_archipelago_proj_gdf.geometry,
        drop=False,
        invert=True
    )

    
    # Create binary mask for all years (1 = seasonal snow, 0 = no seasonal snow)
    modis_binary = xr.where(y_band_clipped >= 56, 1, 0)
    
    # Calculate per-year areas
    band_snow_counts = modis_binary.sum(dim=['x', 'y']).compute()
    
    for wy in modis_seasonal_snow_mask_ds.water_year.values:
        band_area = float(band_snow_counts.sel(water_year=wy).values) * modis_pixel_area_km2
        water_year_areas[wy] += band_area
    
    # Calculate 10-year composite (pixels with ≥3 years of snow)
    years_with_snow = modis_binary.sum(dim='water_year').compute()
    composite_pixels = (years_with_snow >= 3).sum().values
    band_composite_area = float(composite_pixels) * modis_pixel_area_km2
    median_composite_area += band_composite_area
    
    # Progress update
    print(f"  Y-band {i:5d}-{end_idx:5d}: Composite area so far: {median_composite_area:12,.0f} km²")
    
    del y_band_all_years, y_band_clipped, modis_binary, band_snow_counts, years_with_snow


In [ ]:
rows = []
for wy in modis_seasonal_snow_mask_ds.water_year.values:
    rows.append({
        'water_year': int(wy),
        'modis_snow_area_km2': water_year_areas[wy],
    })
    print(f"  WY {wy}: {water_year_areas[wy]:,.0f} km²")

rows.append({
    'water_year': 'composite',
    'modis_snow_area_km2': median_composite_area,
})
print(f"\n10-year median composite area (≥3 years snow): {median_composite_area:,.0f} km²")

coverage_df = pd.DataFrame(rows)
coverage_df

In [ ]:
coverage_df.to_csv(RESULTS_DIR / 'modis_coverage_per_water_year.csv', index=False)

## Get snowmelt runoff onset dataset spatial coverage and temporal resolution

In [ ]:
# temporal_resolution is int16 on disk with scale_factor=0.1, so CF decoding inflates it
# 4x to float64: one 50-row band was 2.2 GB decoded, and .compute() materialised the whole
# band before reducing it — measured peak 8.4 GB per band over populated latitudes, which
# OOM-kills a 16 GB kernel. Read raw int16 instead and let dask reduce lazily (nothing
# bigger than a chunk is ever in memory, peak ~1.6 GB), in shard-aligned slabs rather than
# 50-row bands — the store is sharded 2048x2048, so a 50-row read still pulled whole shards.
# Counts and sums come out identical to the decoded path; raw sums accumulate in int64 and
# are rescaled by scale_factor below. ~19x faster overall: 0.033 vs 0.62 s per latitude row.
LAT_BAND_SIZE = 2048  # = shard height; anything smaller re-reads whole shards

global_ds = config.open_runoff_onset_dataset(
    chunks={'water_year': 1, 'latitude': LAT_BAND_SIZE, 'longitude': 2048},
    mask_and_scale=False,
)

# With decoding off, these stay in .attrs instead of moving to .encoding.
FILL = global_ds['temporal_resolution'].attrs['_FillValue']
TR_SCALE = float(global_ds['temporal_resolution'].attrs['scale_factor'])

# Calculate pixel areas for EPSG:4326
km_per_degree = 111.32

lat_res = abs(float(global_ds.latitude[1] - global_ds.latitude[0]))
lon_res = abs(float(global_ds.longitude[1] - global_ds.longitude[0]))

lats = global_ds.latitude.values

pixel_area_per_lat = (
    (lat_res * km_per_degree) *
    (lon_res * km_per_degree * np.cos(np.deg2rad(lats)))
)

# Checkpoint file
checkpoint_file = RESULTS_DIR / 'runoff_checkpoint.pkl'

# Load or initialize
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'rb') as f:
        checkpoint = pickle.load(f)
    water_year_areas = checkpoint['water_year_areas']
    median_total_area = checkpoint['median_total_area']
    water_year_temporal_res_sum = checkpoint['water_year_temporal_res_sum']
    water_year_temporal_res_count = checkpoint['water_year_temporal_res_count']
    median_temporal_res_sum = checkpoint['median_temporal_res_sum']
    median_temporal_res_count = checkpoint['median_temporal_res_count']
    start_idx = checkpoint['next_idx']
    print(f"Resuming from band {start_idx}")
else:
    water_year_areas = {wy: 0.0 for wy in global_ds.water_year.values}
    median_total_area = 0.0
    water_year_temporal_res_sum = {wy: 0.0 for wy in global_ds.water_year.values}
    water_year_temporal_res_count = {wy: 0 for wy in global_ds.water_year.values}
    median_temporal_res_sum = 0.0
    median_temporal_res_count = 0
    start_idx = 0
    print("Starting fresh")

n_lats = len(lats)

# Bands snap to the shard grid. A checkpoint left mid-shard by an earlier run (e.g. one
# written by the old 50-row loop) just gets a short first band to get back onto it.
bands = []
band_start = start_idx
while band_start < n_lats:
    band_end = min((band_start // LAT_BAND_SIZE + 1) * LAT_BAND_SIZE, n_lats)
    bands.append((band_start, band_end))
    band_start = band_end

print("\n=== Processing runoff onset data ===")

for i, end_idx in tqdm.tqdm(bands):
    pixel_areas_band = pixel_area_per_lat[i:end_idx]

    # temporal_resolution is non-null exactly where runoff_onset is non-null —
    # use it directly as both the valid mask and the value, no need to load runoff_onset.
    # Same for temporal_resolution_median vs runoff_onset_median.
    temporal_res_band = global_ds['temporal_resolution'].isel(latitude=slice(i, end_idx))
    valid_mask_all_years = temporal_res_band != FILL
    median_temporal_res_band = global_ds['temporal_resolution_median'].isel(latitude=slice(i, end_idx))
    median_valid_mask = median_temporal_res_band != FILL

    # All lazy — only these four small reductions ever materialise.
    valid_counts_per_year = valid_mask_all_years.sum(dim='longitude')          # (water_year, latitude)
    tr_sum_per_wy = temporal_res_band.where(valid_mask_all_years, 0).sum(dim=['latitude', 'longitude'])
    median_valid_counts = median_valid_mask.sum(dim='longitude')               # (latitude,)
    median_tr_sum = median_temporal_res_band.where(median_valid_mask, 0).sum()

    valid_counts_per_year, tr_sum_per_wy, median_valid_counts, median_tr_sum = dask.compute(
        valid_counts_per_year, tr_sum_per_wy, median_valid_counts, median_tr_sum
    )

    # --- Spatial coverage + temporal resolution per water year ---
    for wy in global_ds.water_year.values:
        counts_wy = valid_counts_per_year.sel(water_year=wy).values
        water_year_areas[wy] += float((counts_wy * pixel_areas_band).sum())
        water_year_temporal_res_sum[wy] += float(tr_sum_per_wy.sel(water_year=wy)) * TR_SCALE
        water_year_temporal_res_count[wy] += int(counts_wy.sum())

    # --- Median composite spatial coverage + temporal resolution ---
    median_total_area += float((median_valid_counts.values * pixel_areas_band).sum())
    median_temporal_res_sum += float(median_tr_sum) * TR_SCALE
    median_temporal_res_count += int(median_valid_counts.sum())

    # Save checkpoint after every band
    with open(checkpoint_file, 'wb') as f:
        pickle.dump({
            'water_year_areas': water_year_areas,
            'median_total_area': median_total_area,
            'water_year_temporal_res_sum': water_year_temporal_res_sum,
            'water_year_temporal_res_count': water_year_temporal_res_count,
            'median_temporal_res_sum': median_temporal_res_sum,
            'median_temporal_res_count': median_temporal_res_count,
            'next_idx': end_idx
        }, f)

    print(f"  Lat band {i:6d}-{end_idx:6d}: Median area: {median_total_area:12,.0f} km²")

print("\nProcessing complete!")

for wy in global_ds.water_year.values:
    print(f"  WY {wy}: {water_year_areas[wy]:,.0f} km²")
print(f"\nMedian composite area: {median_total_area:,.0f} km²")

os.remove(checkpoint_file)

In [ ]:
rows = []
for wy in global_ds.water_year.values:
    rows.append({
        'water_year': int(wy),
        'runoff_onset_area_km2': water_year_areas[wy],
        'avg_temporal_resolution_days': water_year_temporal_res_sum[wy] / water_year_temporal_res_count[wy],
    })
    print(f"  WY {wy}: {water_year_areas[wy]:,.0f} km²  |  avg temporal res: {water_year_temporal_res_sum[wy] / water_year_temporal_res_count[wy]:.1f} days")

rows.append({
    'water_year': 'composite',
    'runoff_onset_area_km2': median_total_area,
    'avg_temporal_resolution_days': median_temporal_res_sum / median_temporal_res_count,
})
print(f"\nMedian composite area: {median_total_area:,.0f} km²  |  avg temporal res: {median_temporal_res_sum / median_temporal_res_count:.1f} days")

results_df = pd.DataFrame(rows)
results_df

In [ ]:
results_df.to_csv(RESULTS_DIR / 'runoff_onset_coverage_and_temporal_res_per_water_year.csv', index=False)

## Merge results

In [ ]:
modis_df = pd.read_csv(RESULTS_DIR / 'modis_coverage_per_water_year.csv', dtype={'water_year': str})
results_df = pd.read_csv(RESULTS_DIR / 'runoff_onset_coverage_and_temporal_res_per_water_year.csv', dtype={'water_year': str})

results_df = results_df.merge(modis_df, on='water_year', how='left')

results_df['percent_coverage'] = (
    results_df['runoff_onset_area_km2'] / 
    results_df['modis_snow_area_km2'] * 100
)

results_df = results_df[[
    'water_year',
    'runoff_onset_area_km2',
    'modis_snow_area_km2',
    'percent_coverage',
    'avg_temporal_resolution_days'
]]

results_df

In [ ]:
# The excluded-area total is computed by how_much_seasonal_snow_do_we_miss.ipynb
excluded_area_csv = RESULTS_DIR / 'seasonal_snow_excluded_area_summary.csv'
if not excluded_area_csv.exists():
    raise FileNotFoundError(
        f"{excluded_area_csv} not found — run how_much_seasonal_snow_do_we_miss.ipynb "
        f"for {VERSION} first."
    )
seasonal_snow_excluded_area_df = pd.read_csv(excluded_area_csv)
total_excluded_area_km2 = seasonal_snow_excluded_area_df.loc[
    seasonal_snow_excluded_area_df['Region'] == 'Total Excluded Area', 'Snow Area (km^2)'
].values[0]

# Seasonal snow that Sentinel-1 IW VV can never see (Greenland, the Canadian Arctic Archipelago, the
# Russian Arctic Islands, ice-free Antarctica) is missing from the MODIS-observable extent, so add
# it back to get the full seasonal snow extent and report coverage against both denominators.
results_df['total_seasonal_snow_extent_km2'] = (
    results_df['modis_snow_area_km2'] + total_excluded_area_km2
)
results_df = results_df.rename(columns={'percent_coverage': 'percent_coverage_of_modis_snow_area'})
results_df['percent_coverage_of_total_seasonal_snow_extent'] = (
    results_df['runoff_onset_area_km2'] / results_df['total_seasonal_snow_extent_km2'] * 100
)

results_df = results_df[[
    'water_year',
    'runoff_onset_area_km2',
    'modis_snow_area_km2',
    'total_seasonal_snow_extent_km2',
    'percent_coverage_of_modis_snow_area',
    'percent_coverage_of_total_seasonal_snow_extent',
    'avg_temporal_resolution_days'
]]
results_df

In [ ]:
# This is Table 1 in the paper: water year, spatial coverage, seasonal snow extent, coverage,
# average temporal resolution. Table 1's percentages are the
# percent_coverage_of_total_seasonal_snow_extent column.
#
# Up to v9 this was written twice: a base table and a `_REVISED` one. _REVISED was a strict
# superset (same rows and values, percent_coverage renamed to percent_coverage_of_modis_snow_area,
# plus the total-extent column and its percentage) and was the one matching the manuscript, so
# from v10 on only this single table is written.
results_df.to_csv(RESULTS_DIR / 'complete_spatial_coverage_and_temporal_res_per_water_year.csv', index=False)